##### Copyright 2026 Google LLC.

In [2]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: Code Execution

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Code_Execution.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>


The Gemini API [code execution](https://ai.google.dev/gemini-api/docs/code-execution) feature enables the model to generate and run Python code based on plain-text instructions that you give it, and even output graphs. It can learn iteratively from the results until it arrives at a final output.

This notebook is a walk through:
* Understanding how to start using the code execution feature with Gemini API
* Learning how to use code execution on single Gemini API calls
* Running scenarios using local files (or files uploaded to the Gemini File API) via File I/O
* Using code execution on chat interactions
* Performing code execution on multimodal scenarios

> **Note:** This notebook uses the [Interactions API](https://ai.google.dev/gemini-api/docs/interactions), the latest way to interact with Gemini models. Looking for the `generateContent` version? Check the [archive branch](https://github.com/google-gemini/cookbook/blob/archive/generate-content-api/quickstarts/Code_Execution.ipynb).

## Setup

### Install SDK

Install the SDK from [PyPI](https://github.com/googleapis/python-genai).

In [ ]:
%pip install -U -q 'google-genai>=2.0.0'  # 2.0 for Interactions API

### Setup your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) for an example.

In [10]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

### Initialize SDK client

With the new SDK you now only need to initialize a client with your API key (or OAuth if using [Vertex AI](https://cloud.google.com/vertex-ai)). The model is now set in each call.

In [12]:
from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)

Select the model you want to use in this guide:

In [14]:
MODEL_ID = "gemini-3-flash-preview" # @param ["gemini-2.5-flash-preview", "gemini-3.1-flash-lite", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

## Helper function

When using code execution as a tool, the model returns a list of parts including `text`, `executable_code`, `execution_result`, and `inline_data` parts. Use the function below to help you visualize and better display the code execution results. Here are a few details about the different fields of the results:

* `text`: Inline text generated by the model.
* `executable_code`: Code generated by the model that is meant to be executed.
* `code_execution_result`: Result of the `executable_code`.
* `inline_data`: Inline media generated by the model.
> The response from `interactions.create` contains a list of `steps`. For text responses, access the output via `interaction.steps[-1].content[0].text`.


In [16]:
from IPython.display import Image, Markdown, Code, HTML, display
import base64

def display_interaction_result(interaction):
    """Display the outputs of an interaction with code execution."""
    for step in interaction.steps:
        if step.type == "model_output":
            for content in step.content:
                if content.type == "text" and content.text:
                    display(Markdown(content.text))
                elif content.type == "image" and hasattr(content, 'data') and content.data:
                    display(Image(data=base64.b64decode(content.data), width=800, format="jpeg"))
        elif step.type == "code_execution_call":
            code = dict(step.arguments).get('code', '')
            code_html = f'<pre style="background-color: #1e1e1e; color: #d4d4d4; padding: 10px;">{code}</pre>'
            display(HTML(code_html))
        elif step.type == "code_execution_result":
            if step.result:
                display(Markdown(f"**Result:** {step.result}"))
        elif step.type == "thought":
            pass  # Skip thoughts for cleaner output
        display(Markdown("---"))

## Use `code_execution` with a single call

When initiating the model, pass `code_execution` as a `tool` to tell the model that it is allowed to generate and run code.

In [18]:
prompt = """
    What is the sum of the first 50 prime numbers?
    Generate and run code for the calculation, and make sure you get all 50.
"""

interaction = client.interactions.create(
    model=MODEL_ID,
    input=prompt,
    tools=[{"type": "code_execution"}],
)

display_interaction_result(interaction)

<IPython.core.display.Markdown object>
<IPython.core.display.HTML object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>


## Code execution with File I/O

The dataset you will use in this guide comes from the [StatLib](http://lib.stat.cmu.edu/datasets/) from the [Department of Statistics](https://www.cmu.edu/dietrich/statistics-datascience/index.html) at [Carnegie Mellon University](http://www.cmu.edu/). It is made available by the [`scikit-learn`](https://scikit-learn.org) under the 3-Clause BSD license.

It provides 20k information on various blocks in California, including the location (longitute/lattitude), average income,
housing average age, average rooms, average bedrooms, population,
average occupation.

Here's a breakdown of the columns and what the attributes represent:
* MedInc:        median income in block group
* HouseAge:      median house age in block group
* AveRooms:      average number of rooms per household
* AveBedrms:     average number of bedrooms per household
* Population:    block group population
* AveOccup:      average number of household members
* Latitude:      block group latitude
* Longitude:     block group longitude

**Note**: Code execution functionality works best with a `.csv` or `.txt` file.


In [20]:
import pandas as pd
from sklearn.datasets import fetch_california_housing

california_housing = fetch_california_housing(as_frame=True)
california_housing.frame.to_csv('houses.csv', index=False)

In [21]:
# Read the CSV file into a pandas DataFrame
houses_data = pd.read_csv('houses.csv', nrows=5000) # only keeping the first 5000 entries to keep the request light (still 500k tokens). Use pro models to ingest the full dataset.
houses_data.to_csv('houses.csv', index=False)
houses_data.head()

In [22]:
# Upload houses.csv file using the File API
houses_file = client.files.upload(
    file='houses.csv',
    config=types.FileDict(display_name='Blocks Data')
)

print(f"Uploaded file '{houses_file.display_name}' as: {houses_file.uri}")

Uploaded file 'Blocks Data' as: https://generativelanguage.googleapis.com/v1beta/files/iyzivxo1n4f0


Let's try several queries about the dataset that you have. Starting off, it would be interesting to see the most expensive blocks and check wether there's abnomal data.

In [24]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "text", "text": "This dataset provides information on various blocks in California."},
        {"type": "text", "text": "Generate a scatterplot comparing the houses age with the median house value for the top-20 most expensive blocks."},
        {"type": "text", "text": "Use each block as a different color, and include a legend of what each color represents."},
        {"type": "text", "text": "Plot the age as the x-axis, and the median house value as the y-axis."},
        {"type": "text", "text": "In addition, point out on the graph which points could be anomalies? Circle the anomaly in red on the graph. Then save the plot as an image file and display the image."},
        {"type": "document", "uri": houses_file.uri},
    ],
    tools=[{"type": "code_execution"}],
)

display_interaction_result(interaction)

<IPython.core.display.Markdown object>
<IPython.core.display.HTML object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Image object>
<IPython.core.display.Markdown object>
<IPython.core.display.Image object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.HTML object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.HTML object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Image object>
<IPython.core.display.Markdown object>
<IPython.core.display.Image object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>


Moving forward with the data investigation, you can now analyze data variance in the dataset:

In [26]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "text", "text": "This dataset provides information on various blocks in California."},
        {"type": "text", "text": "Analyze how each of the features is correlated to the target variable and find the variance in the dataset."},
        {"type": "text", "text": "Display the results in a heatmap chart and save the chart as an image file."},
        {"type": "document", "uri": houses_file.uri},
    ],
    tools=[{"type": "code_execution"}],
)

display_interaction_result(interaction)

<IPython.core.display.Markdown object>
<IPython.core.display.HTML object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Image object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>


Here is another example - Calculating repeated letters in a word (a common example where LLM sometimes struggle to get the result).

In [28]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input="Calculate how many letter 'r' are in the word 'strawberry'? Generate and run code for the calculation.",
    tools=[{"type": "code_execution"}],
)

In [29]:
display_interaction_result(interaction)

<IPython.core.display.Markdown object>
<IPython.core.display.HTML object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>


## Chat

It works the same when using a `chat`, which allows you to have multi-turn conversations with the model. You can set the `system_instructions` as well, which allows you to further steer the behavior of the model.

In [31]:
system_instruction = """
  You are an expert software developer and a helpful coding assistant.
  You are able to generate high-quality code in any programming language.
"""

# First turn with code execution
turn_1 = client.interactions.create(
    model=MODEL_ID,
    input="Run the bogo-sort algorithm with this list of numbers as input until it completes: [6, 4, 1, 9, 3, 5, 8, 2, 7].",
    system_instruction=system_instruction,
    tools=[{"type": "code_execution"}],
)

This time, you're going to ask the model to use a [Bogo-sort](https://en.wikipedia.org/wiki/Bogosort) algorithm to sort a list of numbers.

In [33]:
display_interaction_result(turn_1)

<IPython.core.display.Markdown object>
<IPython.core.display.HTML object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>


This code seems satisfactory, as it performs the task. However, you can further update the code by sending the following message below the model so that it can mitigate some of the randomness.

In [35]:
turn_2 = client.interactions.create(
    model=MODEL_ID,
    input="Run an alternate implementation of the bogo-sort algorithm with the same list. Compare the efficiency.",
    previous_interaction_id=turn_1.id,
    tools=[{"type": "code_execution"}],
)

display_interaction_result(turn_2)

<IPython.core.display.Markdown object>
<IPython.core.display.HTML object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>


Try running the previous cell multiple times and you'll see a different number of iterations, indicating that the Gemini API indeed ran the code and obtained different results due to the nature of the algorithm.

## Multimodal prompting

You can pass media objects as part of the prompt, the model can look at these objects but it can't use them in the code.

In this example, you will interact with Gemini API, using code execution, to run simulations of the [Monty Hall Problem](https://en.wikipedia.org/wiki/Monty_Hall_problem).

In [39]:
! curl -o montey_hall.png https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/Monty_open_door.svg/640px-Monty_open_door.svg.png

In [40]:
import base64
from IPython.display import Image as IPImage

with open("montey_hall.png", "rb") as f:
    montey_hall_data = base64.b64encode(f.read()).decode('utf-8')

IPImage(filename="montey_hall.png")

In [41]:
prompt="""
    Run a simulation of the Monty Hall Problem with 1,000 trials.

    The answer has always been a little difficult for me to understand when people
    solve it with math - so run a simulation with Python to show me what the
    best strategy is.
"""

# Non-streaming code execution
interaction = client.interactions.create(
    model=MODEL_ID,
    input=prompt,
    tools=[{"type": "code_execution"}],
)

display_interaction_result(interaction)

<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>


## Streaming

Streaming is compatible with code execution, and you can use it to deliver a response in real time as it gets generated. Just note that successive parts of the same type (`text`, `executable_code` or `execution_result`) are meant to be joined together, and you have to stitch the output together yourself:

In [44]:
# Streaming with code execution
from IPython.display import Markdown, HTML

for event in client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "text", "text": prompt},
        {"type": "image", "data": montey_hall_data, "mime_type": "image/jpeg"},
    ],
    tools=[{"type": "code_execution"}],
    stream=True,
):
    if hasattr(event, 'delta'):
        if hasattr(event.delta, 'text') and event.delta.text:
            print(event.delta.text, end="")

BadRequestError: event: error
data: {"error":{"message":"Unable to process input image. Please retry or report in https://developers.generativeai.google/guide/troubleshooting","code":"invalid_request"},"event_type":"e

## Next Steps
### Useful API references:

Check the [Code execution documentation](https://ai.google.dev/gemini-api/docs/code-execution) for more details about the feature and in particular, the [recommendations](https://ai.google.dev/gemini-api/docs/code-execution?lang=python#code-execution-vs-function-calling) regarding when to use it instead of [function calling](https://ai.google.dev/gemini-api/docs/function-calling).

### Continue your discovery of the Gemini API

Please check other guides from the [Cookbook](https://github.com/google-gemini/cookbook/) for further examples on how to use Gemini and in particular [this example](../quickstarts/Get_started_LiveAPI_tools.ipynb) showing how to use the different tools (including code execution) with the Live API.

The [Search grounding](./Search_Grounding.ipynb) guide also has an example mixing grounding and code execution that is worth checking.

To see how code execution is used with Gemini 1.5, please take a look at the [legacy code execution example](https://github.com/google-gemini/cookbook/blob/gemini-1.5-archive/quickstarts/Code_Execution.ipynb).